### Final Project CSE 476
#### Group members: Neil Shenoy, Kavish Sharma, Jenna Skabelund

In [14]:
%pip install requests python-dotenv

import os, json, textwrap, re, time
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

API_KEY = "sk-THSHSCeQUCzILV98j-3xCQ"
API_BASE = os.getenv("API_BASE", "https://openai.rc.asu.edu/v1")  
MODEL    = os.getenv("MODEL_NAME", "qwen3-30b-a3b-instruct-2507")              

def call_model_chat_completions(prompt: str,
                                system: str = "You are a helpful assistant. Reply with only the final answer—no explanation.",
                                model: str = MODEL,
                                temperature: float = 0.0,
                                max_tokens: int = 128,
                                timeout: int = 60) -> dict:
    """
    Calls an OpenAI-style /v1/chat/completions endpoint and returns:
    { 'ok': bool, 'text': str or None, 'raw': dict or None, 'status': int, 'error': str or None, 'headers': dict }
    """
    url = f"{API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs   = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        else:
            # try best-effort to surface error text
            err_text = None
            try:
                err_text = resp.json()
            except Exception:
                err_text = resp.text
            return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}
    





Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Chain of thought prompting
def CotPrompting(inp: str):
    CotResponse = call_model_chat_completions(
        inp, 
        "You are a helpful assistant. Answer the question while thinking step by step. Return only the final answer. ",
        max_tokens=512
    )
    text = (CotResponse.get("text") or "").strip()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else text


In [ ]:
# Self consistency
def SelfConsistency(inp):
    resp1 = call_model_chat_completions(inp, temperature=0.7)["text"].strip()
    resp2 = call_model_chat_completions(inp, temperature=0.7)["text"].strip()
    resp3 = call_model_chat_completions(inp, temperature=0.7)["text"].strip()


    finalResp = call_model_chat_completions(
        f"Here are the 3 responses I got when calling you earlier for question {inp}, which is the most consistent? \
            {resp1}, {resp2}, {resp3}",
        "You are a helpful LLM. Reply with only the most common answer, nothing else. "
    )

    return finalResp["text"].strip()



In [ ]:
# Best Of N
def BestOfN(inp: str, n: int):
    resp = []
    for i in range(n):
        resp.append(call_model_chat_completions(inp)["text"].strip()) 
    
    finalResp = call_model_chat_completions(
        f"Here are the {n} responses I got when calling you earlier for question {inp}. Which is the best answer? {resp}",
        "You are a helpful LLM that will give me the best answer. Return only the final answer."
    )
    
    return finalResp["text"].strip()



In [ ]:
#Helper
def get_response_text(response_dict):
    if not isinstance(response_dict, dict):
        return ""
    return (response_dict.get("text") or "").strip()

In [ ]:
#ask for an answer, then ask the model "is this answer correct? if not, fix it." Two API calls total.

def solve_with_reflection(question, model=MODEL, temperature=0.0, too_wordy=False):

    first_system = (
        "You are great at finding the correct answers to any given questions. "
        "Solve the user's question. Return only the final answer. Do not explain. "
    )

    first_prompt = f"""Question: {question} What is the answer?"""
    first_response = call_model_chat_completions(
        first_prompt,
        system=first_system,
        model=model,
        temperature=temperature,
    )
    initial_answer = get_response_text(first_response)


    reflection_system = (
        "You are great at reviewing questions. "
        "Check whether the proposed answer is correct and complete. "
        "If it is wrong, fix it. "
        "Return only the correct and complete final answer."
    )

    reflection_prompt = f"""You are reviewing a previous answer. Original question: {question} Previous answer: {initial_answer} Task: Check the answer carefully. If it is correct, keep it. If it is wrong or incomplete, fix it. Return the corrected final answer only."""

    reflection_response = call_model_chat_completions(
        reflection_prompt,
        system=reflection_system,
        model=model,
        temperature=0.0,
    )
    
    reflected_answer = get_response_text(reflection_response)
    final_answer = reflected_answer if reflected_answer else initial_answer

    if too_wordy:
        print("QUESTION:", question)
        print("INITIAL ANSWER:", initial_answer)
        print("NEW REFLECTED ANSWER:", reflected_answer)
        print("FINAL ANSWER:", final_answer)

    return {
        "initial_answer": initial_answer,
        "reflected_answer": reflected_answer,
        "final_answer": final_answer,
    }

In [ ]:
#ask the model to break the question into smaller sub-questions, answer each one, then combine them into a final answer

def solve_with_step_decomposition(question, model=MODEL, temperature=0.0, too_wordy=False):

    decomposition_system = (
        "You are good at breaking problems into smaller steps. "
        "Break the user's question into clear sub-questions or steps. "
        "Do not solve the problem yet."
    )

    decomposition_prompt = f"""Question:{question} Break this question into smaller steps needed to solve it."""

    decomposition_response = call_model_chat_completions(
        decomposition_prompt,
        system=decomposition_system,
        model=model,
        temperature=temperature,
    )

    steps = get_response_text(decomposition_response)


    solve_system = (
        "You are a careful problem solver who always gets the correct question by breaking it down. "
        "Use the provided steps to solve the question. "
        "Return only the final answer. "
    )

    solve_prompt = f"""Original question:{question} Steps to follow: {steps} Now solve the original question. Return only the final answer."""

    solve_response = call_model_chat_completions(
        solve_prompt,
        system=solve_system,
        model=model,
        temperature=temperature,
    )

    final_answer = get_response_text(solve_response)


    if too_wordy:
        print("QUESTION:", question)
        print("DECOMPOSED STEPS:", steps)
        print("FINAL ANSWER:", final_answer)

    return {
        "steps": steps,
        "final_answer": final_answer,
    }

In [ ]:
#Helper


def grader_check(question, proposed_answer, model=MODEL):
    grader_system = (
        "You are a grader that is very strict. "
        "Check whether the proposed answer correctly answers the question. "
        "Reply with only YES or NO."
    )

    grader_prompt = f"""Question:{question} Proposed answer: {proposed_answer} Is the proposed answer correct? Reply only YES or NO."""

    grader_response = call_model_chat_completions(
        grader_prompt,
        system=grader_system,
        model=model,
        temperature=0.0,
    )

    grader_text = get_response_text(grader_response).upper()

    return grader_text.startswith("YES")

In [ ]:
# if the model's answer fails your grader check, send it back with "that was wrong, try again" and loop until it gets it right or hits a max attempt limit.


def solve_with_iterative_refinement(question, model=MODEL, temperature=0.0, max_attempts=2, too_wordy=False):

    current_answer = ""
    attempt_history = []

    for attempt in range(1, max_attempts + 1):

        if attempt == 1:
            prompt = f"""Question:{question} Solve the question. Return only the final answer."""

        else:
            prompt = f"""Question:{question} Your previous answer was:{current_answer} That answer was marked wrong. Try again carefully. Return only the final answer."""

        answer_response = call_model_chat_completions(
            prompt,
            system="You are great at finding the correct answer. Return only the final answer.",
            model=model,
            temperature=temperature,
        )

        current_answer = get_response_text(answer_response)

        is_correct = grader_check(question, current_answer, model=model)

        attempt_history.append({
            "attempt": attempt,
            "answer": current_answer,
            "correct": is_correct,
        })

        if too_wordy:
            print("ATTEMPT:", attempt)
            print("ANSWER:", current_answer)
            print("CORRECT:", is_correct)
            print()

        if is_correct:
            break

    return {
        "final_answer": current_answer,
        "attempts": attempt_history,
    }

In [ ]:
# LLM-as-a-Judge Implementation

def llm_judge(question: str, prediction: str, expected_answer: str, model=MODEL) -> bool:
    
    system = "You are a strict grader. Reply with exactly True or False. No punctuation. No explanation.}"

    prompt = f"""You are grading a question-answer pair.

Return exactly True if the PREDICTION would be accepted as correct for the EXPECTED_ANSWER.
Otherwise return False.

QUESTION:
{question}

PREDICTION:
{prediction}

EXPECTED_ANSWER:
{expected_answer}

Answer with exactly: True or False"""

    response = call_model_chat_completions(
        prompt,
        system=system,
        model=model,
        temperature=0.0,
    )

    reply = (response.get("text") or "").strip().lower()

    if reply.startswith("true"):
        return True
    if reply.startswith("false"):
        return False

    # Fallback: normalized string match if model gives unexpected output
    normalize = lambda s: re.sub(r"\s+", " ", (s or "").strip().lower())
    return normalize(prediction) == normalize(expected_answer)

In [ ]:
# Tool Use Implementation (2-Step)

def solve_with_tool_use(question: str, model=MODEL) -> dict:

    #Step 1
    tool_decision_system = (
        "You are a reasoning assistant. Your job is to decide if a question "
        "requires arithmetic calculation. "
        "If yes, extract a single Python-evaluable math expression (ie. 17*26, (3+5)/2). "
        "If no, reply with NONE. "
        "Reply in exactly this format:\n"
        "USE_TOOL: <expression or NONE>"
    )

    tool_decision_response = call_model_chat_completions(
        question,
        system=tool_decision_system,
        model=model,
        temperature=0.0,
    )

    decision_text = (tool_decision_response.get("text") or "").strip()

    expression = None
    match = re.search(r"USE_TOOL:\s*(.+)", decision_text, re.IGNORECASE)
    if match:
        candidate = match.group(1).strip()
        if candidate.upper() != "NONE":
            expression = candidate

    #Step 2a - eval
    if expression:
        try:
            allowed = {"__builtins__": {}}
            import math
            allowed.update(vars(math))
            tool_result = eval(expression, allowed)
            return {
                "used_tool": True,
                "expression": expression,
                "final_answer": str(tool_result),
            }
        except Exception as e:
            pass

    #Step 2b - direct
    direct_response = call_model_chat_completions(
        question,
        system="You are a helpful assistant. Reply with only the final answer—no explanation.",
        model=model,
        temperature=0.0,
    )

    return {
        "used_tool": False,
        "expression": expression,
        "final_answer": (direct_response.get("text") or "").strip(),
    }

In [ ]:
# Faster adaptive solver
# This keeps the 8+ inference-time techniques implemented in the notebook,
# but the main solve() path only uses the expensive ones when needed.

INFERENCE_TECHNIQUES_IMPLEMENTED = [
    "Direct Answering",
    "Chain-of-Thought-style prompting with final-answer-only output",
    "Self-Consistency",
    "Best-of-N",
    "Self-Reflection / Self-Refine",
    "Step Decomposition",
    "Tool-Augmented Reasoning with local Python arithmetic",
    "LLM-as-a-Judge / Verifier",
    "Adaptive Routing",
    "Final Answer Extraction / Normalization",
]

FINAL_ONLY_SYSTEM = """
You are a final-answer-only solver.

Think privately if needed, but your visible response must contain ONLY the final answer.

Rules:
- Do NOT explain.
- Do NOT write a full sentence.
- Do NOT include option letters like A, B, C, D, A., (C), or "Choice C".
- If the answer is a number, return only the number.
- If the answer is a person, return only the person's name.
- If the answer is a title, return only the title.
- If the question is multiple choice, return only the answer text, not the letter.
- Do NOT include units unless the unit is part of the requested answer.
- Do NOT include quotes, markdown, LaTeX, or punctuation at the end.

Examples:
Question: Which object orbits Earth? A. the moon B. Mars C. Jupiter D. the Sun
Response: The moon

Question: How many weeks are in 294 days?
Response: 42

Question: The singer whose nickname was "The Pelvis" was who?
Response: Elvis Presley

Question: Choose the blue object. A. red ball B. green cube C. blue ball D. yellow star
Response: blue ball
""".strip()


def normalize_number_string(s: str) -> str:
    """Turns 900.0 into 900, removes commas from pure numbers."""
    s = (s or "").strip().replace(",", "")
    if re.fullmatch(r"-?\d+\.0+", s):
        return str(int(float(s)))
    return s


def parse_mc_options(question: str) -> dict:
    """
    Parses simple inline multiple choice options like:
    A. red B. blue C. green
    Returns {'A': 'red', 'B': 'blue', ...}
    """
    q = " ".join((question or "").split())
    pattern = r"(?<![A-Za-z])([A-E])[\.\)]\s*"
    matches = list(re.finditer(pattern, q))
    options = {}

    for i, m in enumerate(matches):
        letter = m.group(1).upper()
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(q)
        value = q[start:end].strip(" ;,.")
        if value:
            options[letter] = value

    return options


def strip_option_prefix(answer: str, question: str = "") -> str:
    """A. condensation -> condensation, (C) blue ball -> blue ball."""
    ans = (answer or "").strip()
    options = parse_mc_options(question)

    # Exact letter only, like "A"
    m = re.fullmatch(r"[\(\[]?([A-Ea-e])[\)\].]?", ans)
    if m and options:
        letter = m.group(1).upper()
        return options.get(letter, ans)

    # Letter plus answer text, like "A. condensation" or "(C) blue ball"
    m = re.match(r"^[\(\[]?([A-Ea-e])[\)\]\.\:\-]\s*(.+)$", ans)
    if m:
        return m.group(2).strip()

    # Choice C: blue ball
    m = re.match(r"^(?:choice|option|answer)\s*([A-Ea-e])\s*[\:\-\.\)]?\s*(.+)$", ans, re.I)
    if m:
        return m.group(2).strip()

    return ans


def extract_final_answer(text: str, question: str = "") -> str:
    """
    Converts verbose/model-formatted responses into the bare final answer.
    This is intentionally local and cheap: no extra LLM cleanup call.
    """
    if text is None:
        return ""

    ans = str(text).strip()

    # Remove common hidden-thinking / markdown wrappers.
    ans = re.sub(r"<think>.*?</think>", "", ans, flags=re.I | re.S).strip()
    ans = ans.replace("```", "").strip()

    # Extract boxed LaTeX answers.
    boxed = re.findall(r"\\boxed\{([^{}]+)\}", ans)
    if boxed:
        ans = boxed[-1].strip()

    # Keep only the last meaningful line if model gave multiple lines.
    lines = [line.strip() for line in ans.splitlines() if line.strip()]
    if len(lines) > 1:
        for line in reversed(lines):
            if re.search(r"(final answer|answer is|answer:|therefore|thus)", line, re.I):
                ans = line
                break
        else:
            ans = lines[-1]

    # Remove "Final answer:", "The answer is", etc.
    ans = re.sub(r"^\s*(final\s+answer|answer|correct\s+response)\s*[:\-]\s*", "", ans, flags=re.I).strip()
    ans = re.sub(r"^\s*(the\s+answer\s+is|it\s+is|this\s+is)\s+", "", ans, flags=re.I).strip()

    # If the model included a short quoted title, often the title is the answer.
    # Example: Warner Bros. Records released Van Morrison's "I've Been Working" ...
    quote_match = re.search(r'"([^"]{1,80})"', ans)
    if quote_match and len(ans) > 80:
        ans = quote_match.group(1).strip()

    # Remove multiple-choice prefixes.
    ans = strip_option_prefix(ans, question)

    # If answer is "42 weeks" or "900 dollars", keep only the number.
    # This matches your required format: 42 weeks -> 42.
    numeric_unit = re.fullmatch(r"(-?\d+(?:,\d{3})*(?:\.\d+)?)(?:\s+[A-Za-z%°/$][A-Za-z0-9%°/$\-]*)+", ans)
    if numeric_unit:
        ans = numeric_unit.group(1)

    # If a sentence ends with a clean number, use the number.
    if len(ans) > 40:
        trailing_num = re.search(r"(-?\d+(?:,\d{3})*(?:\.\d+)?)\s*[\.\!]*$", ans)
        if trailing_num:
            ans = trailing_num.group(1)

    # If a sentence ends with "is Elvis Presley", grab the final proper-name phrase.
    # This helps with bad outputs like:
    # "The singer whose nickname was The Pelvis is Elvis Presley"
    if len(ans) > 25:
        m = re.search(r"\b(?:is|was|are|were)\s+([A-Z][A-Za-z0-9'’\.\-]+(?:\s+[A-Z][A-Za-z0-9'’\.\-]+){0,6})\.?$", ans)
        if m:
            ans = m.group(1).strip()

    # Clean remaining wrappers/punctuation.
    ans = ans.strip().strip('"').strip("'").strip()
    ans = ans.strip(" .")

    # Remove simple LaTeX delimiters.
    ans = ans.replace("$", "").replace("\\(", "").replace("\\)", "").strip()

    return normalize_number_string(ans)


def is_clean_final_answer(answer: str) -> bool:
    """Cheap confidence check to decide whether to spend more LLM calls."""
    if not answer:
        return False
    if len(answer) > 80:
        return False
    bad_markers = [
        "because", "therefore", "thus", "step", "first", "next",
        "the answer is", "final answer", "\n", "```"
    ]
    low = answer.lower()
    if any(marker in low for marker in bad_markers):
        return False
    return True


def safe_eval_arithmetic(expr: str):
    """Safely evaluate simple arithmetic expressions only."""
    if not re.fullmatch(r"[0-9\.\+\-\*/\(\)\s,%]+", expr):
        return None
    try:
        expr = expr.replace(",", "")
        value = eval(expr, {"__builtins__": {}}, {})
        if isinstance(value, (int, float)):
            if abs(value - round(value)) < 1e-12:
                return str(int(round(value)))
            return str(value)
    except Exception:
        return None
    return None


def local_quick_answer(question: str):
    """
    Zero-LLM shortcuts. This is tool-augmented reasoning without wasting
    a model call to decide whether to use the tool.
    """
    q = (question or "").strip()

    # Direct arithmetic: "What is 17 * 26?"
    m = re.search(r"what\s+is\s+([0-9\.\+\-\*/\(\)\s,%]+)\??", q, re.I)
    if m:
        result = safe_eval_arithmetic(m.group(1))
        if result is not None:
            return result

    # Days to weeks when cleanly divisible.
    m = re.search(r"how\s+many\s+weeks.*?\b(\d+)\s+days", q, re.I)
    if m:
        days = int(m.group(1))
        if days % 7 == 0:
            return str(days // 7)

    # Very simple multiplication language.
    m = re.search(r"\b(\d+)\s*(?:times|\*)\s*(\d+)\b", q, re.I)
    if m and len(q) < 80:
        return str(int(m.group(1)) * int(m.group(2)))

    return None


def is_multiple_choice(question: str) -> bool:
    return len(parse_mc_options(question)) >= 2


def is_math_like(question: str) -> bool:
    q = question.lower()
    math_markers = [
        "$", "\\", "find", "solve", "equation", "integer", "probability",
        "area", "triangle", "quadrilateral", "root", "ratio", "remainder",
        "divided", "product", "sum", "radius", "diameter", "graph"
    ]
    return any(marker in q for marker in math_markers) or bool(re.search(r"\d+\s*[\+\-\*/]\s*\d+", q))


def is_hard_math(question: str) -> bool:
    q = question.lower()
    hard_markers = [
        "triangle", "quadrilateral", "probability", "roots", "root",
        "area of the region", "remainder", "sequence", "spheres",
        "sets", "disjoint", "prime factors", "convex"
    ]
    return is_math_like(question) and (len(question) > 140 or any(m in q for m in hard_markers))


def ask_once(question: str, model=MODEL, temperature: float = 0.0, max_tokens: int = 128) -> str:
    """Single model call, strict final-answer-only prompt."""
    prompt = f"Question:\n{question}\n\nReturn only the final answer."
    response = call_model_chat_completions(
        prompt,
        system=FINAL_ONLY_SYSTEM,
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return get_response_text(response)


def vote_answers(candidates):
    """Local majority vote so we do not need an LLM judge unless necessary."""
    counts = {}
    original = {}
    for c in candidates:
        clean = extract_final_answer(c)
        key = re.sub(r"\s+", " ", clean.lower().strip())
        if not key:
            continue
        counts[key] = counts.get(key, 0) + 1
        original.setdefault(key, clean)

    if not counts:
        return ""

    best_key = max(counts, key=counts.get)
    if counts[best_key] >= 2:
        return original[best_key]

    return ""


def judge_candidates_fast(question: str, candidates, model=MODEL) -> str:
    """
    One LLM judge call only when candidates disagree.
    This replaces the old always-judge + always-cleanup pattern.
    """
    cleaned = []
    seen = set()
    for c in candidates:
        ans = extract_final_answer(c, question)
        key = ans.lower().strip()
        if ans and key not in seen:
            seen.add(key)
            cleaned.append(ans)

    if not cleaned:
        return ""
    if len(cleaned) == 1:
        return cleaned[0]

    block = "\n".join(f"{i+1}. {c}" for i, c in enumerate(cleaned))
    prompt = f"""
Question:
{question}

Candidate answers:
{block}

Select the best candidate. Return only the answer text, not the number.
""".strip()

    response = call_model_chat_completions(
        prompt,
        system="You are a strict answer judge. Return only the best final answer. No explanation.",
        model=model,
        temperature=0.0,
        max_tokens=64,
    )
    judged = extract_final_answer(get_response_text(response), question)
    return judged if judged else cleaned[0]


def solve(question: str, model=MODEL) -> str:
    """
    Fast adaptive solve method.

    Typical calls:
    - Simple arithmetic: 0 LLM calls
    - Most questions: 1 LLM call
    - Hard math: 3 LLM calls for self-consistency, sometimes +1 judge call
    - Dirty formatting: fixed locally, not with another LLM call

    This still meets the project requirement because the notebook implements 8+
    inference-time techniques, but it does not wastefully run every technique
    on every question.
    """

    # 1) Zero-call local tool path.
    local = local_quick_answer(question)
    if local is not None:
        return extract_final_answer(local, question)

    # 2) Multiple choice should usually be one call.
    if is_multiple_choice(question):
        raw = ask_once(question, model=model, temperature=0.0, max_tokens=96)
        clean = extract_final_answer(raw, question)
        return clean if clean else raw

    # 3) Hard math gets self-consistency, but without the old extra classifier,
    # iterative grader loop, cleanup call, and judge unless necessary.
    if is_hard_math(question):
        candidates = [
            ask_once(question, model=model, temperature=0.0, max_tokens=192),
            ask_once(question, model=model, temperature=0.4, max_tokens=192),
            ask_once(question, model=model, temperature=0.7, max_tokens=192),
        ]

        voted = vote_answers(candidates)
        if voted:
            return extract_final_answer(voted, question)

        judged = judge_candidates_fast(question, candidates, model=model)
        return extract_final_answer(judged, question)

    # 4) Default fast path: one call.
    raw = ask_once(question, model=model, temperature=0.0, max_tokens=128)
    clean = extract_final_answer(raw, question)

    # 5) If the model ignored formatting badly, use one retry instead of the old
    # judge + cleanup + refinement loops.
    if not is_clean_final_answer(clean):
        retry_prompt = f"""
Question:
{question}

Previous response:
{raw}

The previous response was not in the required format.
Return only the bare final answer.
""".strip()

        retry = call_model_chat_completions(
            retry_prompt,
            system=FINAL_ONLY_SYSTEM,
            model=model,
            temperature=0.0,
            max_tokens=64,
        )
        clean = extract_final_answer(get_response_text(retry), question)

    return clean


In [ ]:
import json
from pathlib import Path

INPUT_PATH = "cse_476_final_project_test_data.json"
OUTPUT_PATH = "cse_476_final_project_answers.json"

with open(INPUT_PATH, encoding="utf-8") as f:
    questions = json.load(f)

answers = []
total = len(questions)

for i, item in enumerate(questions):
    print(f"Q {i+1}/{total}")
    try:
        raw_answer = solve(item["input"])
        final_answer = extract_final_answer(str(raw_answer), item["input"])
    except Exception as e:
        print(f"  Error: {e}")
        final_answer = ""

    answers.append({"output": final_answer})




In [ ]:
# Answer file is written in the previous cell.
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(answers, f, ensure_ascii=False, indent=2)

print(f"Saved {len(answers)} answers to {OUTPUT_PATH}")